# Lab 4: Gradient Descent

Training a model means finding parameter values that make a loss (a measure of the model's error) small. Gradient descent is the loop that does it: repeatedly step against the gradient. In this lab you code it, run it on functions simple enough to see what happens, watch it fail in several ways, and then use it on a model.

1. Derivatives and descent: implement a numerical derivative and gradient descent.
2. Typical hurdles: local minima, vanishing and exploding gradients.
3. Higher dimensions: a numerical gradient and ill-conditioned curvature.
4. Optimizing parameters: fit a one-parameter model to data.
5. Classification: train a sigmoid neuron and compare it to the perceptron.

Every gradient is derived by hand and checked against a numerical estimate: `numerical_derivative` for one variable (part 1), `numerical_gradient` for several (part 3).

## Learning objectives

By the end of this lab, you should be able to:
- Implement gradient descent, a numerical derivative and a numerical gradient
- Reproduce and explain how gradient descent fails: learning rate too large or too small, local minima, vanishing and exploding gradients, ill-conditioned curvature
- Fit the parameter of a model by gradient descent
- Train a sigmoid neuron with stochastic gradient descent and compare it to the perceptron from Lab 3


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import sys
sys.path.append("..")

np.random.seed(0)

---
## 1. Derivatives and descent

Gradient descent needs the gradient (derivative) of the function being minimized. A wrong gradient raises no error, training just quietly goes wrong, so hand-derived gradients are always checked against a numerical estimate, the **central difference**:

$$f'(x) \approx \frac{f(x + h) - f(x - h)}{2h}$$

for a small `h`.

**Exercise:** implement `numerical_derivative(f, x, h=1e-5)`.


In [ ]:
def numerical_derivative(f, x, h=1e-5):
    # your code here
    ...


# self-check: the derivative of x^2 is 2x
for x in [-2.0, 0.0, 1.0, 3.5]:
    approx = numerical_derivative(lambda x: x ** 2, x)
    expected = 2 * x
    assert abs(approx - expected) < 1e-4, f"at x={x}, expected ~{expected}, got {approx}"

print("numerical_derivative() looks correct on these cases.")

Gradient descent starts at a point `x0` and repeatedly steps against the gradient:

$$x_{t+1} = x_t - \text{lr} \cdot \nabla f(x_t)$$

where `lr` is the **learning rate**, which scales the step. The sequence of points visited is the **trajectory**.

**Exercise:** implement `compute_gradient_trajectory(grad_f, x0, lr, n_steps)`, which runs `n_steps` steps and returns the full trajectory, so the descent can be plotted: a list of the `n_steps + 1` points visited, starting with `x0`.

`x0` may be a number or a NumPy array; the same code must handle both.


In [ ]:
def compute_gradient_trajectory(grad_f, x0, lr, n_steps):
    # your code here
    ...


**Exercise:** consider $f(x) = (x - 3)^2$, which has its minimum at `x = 3`. Derive its gradient by hand and implement it as `grad_quad(x)`, then check it against `numerical_derivative`.


In [ ]:
def f_quad(x):
    return (x - 3) ** 2


def grad_quad(x):
    # your code here
    ...


# self-check
for x in [-5.0, 0.0, 3.0, 10.0]:
    approx = numerical_derivative(f_quad, x)
    assert abs(approx - grad_quad(x)) < 1e-4, f"at x={x}, expected ~{approx}, got {grad_quad(x)}"

print("grad_quad() looks correct on these cases.")

### Visualizing a trajectory in 1D

`plot_trajectory_1d` (given) draws a function of one variable as a curve, with a trajectory on top. It draws on the current axes, so call `plt.show()` afterwards. For example:

```python
plot_trajectory_1d(f_quad, trajectory)
plt.show()
```


In [ ]:
def plot_trajectory_1d(f, trajectory, xlim=None):
    """Draw f as a curve, with a trajectory on top (the points visited, joined in order).

    Draws on the current axes.

    f          : function of a number, returning a number
    trajectory : list of numbers, as returned by compute_gradient_trajectory
    xlim       : plotted range (default: a range around the trajectory)
    """
    traj = np.array(trajectory, dtype=float)
    if xlim is None:
        margin = 0.1 * (traj.max() - traj.min()) + 1.0
        xlim = (traj.min() - margin, traj.max() + margin)

    xs = np.linspace(xlim[0], xlim[1], 400)
    plt.plot(xs, [f(x) for x in xs], color="gray")
    plt.plot(traj, [f(x) for x in traj], "o-", color="red", markersize=3, linewidth=1)
    plt.plot(traj[0], f(traj[0]), "s", color="green", markersize=8, label="start")
    plt.plot(traj[-1], f(traj[-1]), "X", color="red", markersize=10, label="end")
    plt.xlim(xlim)
    plt.xlabel("x")
    plt.ylabel("f(x)")
    plt.legend()


**Exercise:** run `compute_gradient_trajectory` on `f_quad` from `x0=0.0` with `lr=0.1` for 50 steps, then visualize the trajectory with `plot_trajectory_1d`.


In [ ]:
# your code here

### Learning rate sweep

The cell below runs the same descent with several learning rates.

**Discussion question:** What happens when the learning rate grows too large?


In [ ]:
learning_rates = [0.01, 0.1, 0.5, 0.9, 1.05]

plt.figure(figsize=(12, 7))
for i, lr in enumerate(learning_rates):
    trajectory = compute_gradient_trajectory(grad_quad, x0=0.0, lr=lr, n_steps=30)
    plt.subplot(2, 3, i + 1)
    plot_trajectory_1d(f_quad, trajectory)
    plt.title(f"lr={lr}")
plt.tight_layout()
plt.show()


---
## 2. Typical gradient descent hurdles

Part 1 used the simplest possible function. Gradient descent only sees the slope at the current point: it cannot know about other minima, or how far away the minimum is. In deep learning nobody can see the loss landscape, so this is the normal situation. The two landscapes below show how it goes wrong.

### Local minima and initialization

`f_multi` has two minima of different depths. In deep learning, parameters start at random values, and different starting values can end in different solutions.


In [ ]:
def f_multi(x):
    return x ** 4 - 0.5 * x ** 3 - 3 * x ** 2


xs = np.linspace(-4, 4, 400)
plt.plot(xs, f_multi(xs))
plt.xlabel("x")
plt.ylabel("f(x)")
plt.title("A function with several local minima")
plt.show()

**Exercise:** derive the gradient of `f_multi` by hand, implement it as `grad_multi(x)`, and check it against `numerical_derivative`.

In [ ]:
def grad_multi(x):
    # your code here
    ...


# self-check
for x in [-3.0, -1.0, 0.5, 2.0]:
    approx = numerical_derivative(f_multi, x)
    assert abs(approx - grad_multi(x)) < 1e-3, f"at x={x}, expected ~{approx}, got {grad_multi(x)}"

print("grad_multi() looks correct on these cases.")

**Exercise:** sweep many starting points across the domain shown above, run gradient descent from each, and make a scatter plot with the starting point on the x-axis and the final `x` reached on the y-axis.


In [ ]:
# your code here

**Discussion question:** What happens if you start exactly at the top of the bump? Why?

### Vanishing and exploding gradients

`f_steep` is nearly flat far from its minimum at `x=0`, and its slope grows without bound as `x` approaches 0. In deep networks, gradients can likewise become too small to make progress or too large to stay stable (the cause there is different: gradients are multiplied through many layers), but the symptom in training is the same.


In [ ]:
def f_steep(x):
    return np.sqrt(np.abs(x)) / (1 + np.sqrt(np.abs(x)))


xs = np.linspace(-10, 10, 400)
plt.plot(xs, f_steep(xs))
plt.xlabel("x")
plt.ylabel("f(x)")
plt.title("Flat far from the minimum, steep close to it")
plt.show()


**Exercise:** derive the gradient of `f_steep` by hand and implement it as `grad_steep(x)`, checked against `numerical_derivative`.

In [ ]:
def grad_steep(x):
    # your code here
    ...


# self-check
for x in [-3.0, -1.0, -0.1, 0.1, 0.5, 1.0, 3.0]:
    approx = numerical_derivative(f_steep, x)
    assert abs(approx - grad_steep(x)) < 1e-3, f"at x={x}, expected ~{approx}, got {grad_steep(x)}"

print("grad_steep() looks correct on these cases.")


**Exercise:** run `compute_gradient_trajectory` on `f_steep` from `x0=3.0` for 200 steps, with a few different learning rates. Visualize the trajectory for each with `plot_trajectory_1d`.

**Discussion question:** is there a learning rate that gets close to the minimum within 200 steps and stays there? Explain what you observe.


In [ ]:
# your code here

---
## 3. Higher dimensions

Training a deep learning model with gradient descent means minimizing a loss over all of the model's parameters at once: millions to billions of numbers, so the landscape has that many dimensions. We can't picture it, so we study a function where we control the geometry: a quadratic bowl in `d` dimensions, with a different steepness along each axis. At `d=2` you can draw the descent. For larger `d` you can only watch the function value over the steps, the same curve you will watch when training real models.

### A quadratic in d dimensions

The function below is $f(x) = \sum_i \lambda_i x_i^2$, with a different curvature $\lambda_i$ (`eigenvalues[i]`) along each axis. `make_eigenvalues(d)` spreads the $\lambda_i$ from `0.1` up to `d / 2`. When the curvatures differ a lot, the function is called **ill-conditioned**.


In [ ]:
def make_eigenvalues(d, lambda_min=0.1, lambda_max=None):
    if lambda_max is None:
        lambda_max = d / 2
    return np.logspace(np.log10(lambda_min), np.log10(lambda_max), d)


def f_quadratic(x, eigenvalues):
    return np.sum(eigenvalues * x ** 2)

### A numerical gradient

For several variables, the numerical estimate is computed one coordinate at a time: the i-th entry of the gradient is the central difference of $t \mapsto f(x + t\,e_i)$ at $t = 0$, where $e_i$ is the i-th unit vector.

In `minitorch.utils`, you were provided a function `numerical_gradient(f, x, h=1e-5)` that does precisely that. You are encoraged to have a look and read how it works later, but for now, you can simply use it.

In [ ]:
from minitorch.utils import numerical_gradient

**Exercise:** derive the gradient of `f_quadratic` with respect to `x` by hand, implement it as `grad_quadratic(x, eigenvalues)`, and check it against `numerical_gradient`.


In [ ]:
def grad_quadratic(x, eigenvalues):
    # your code here
    ...


# self-check, d=3
eigenvalues_check = np.array([0.5, 2.0, 10.0])
x_check = np.array([1.0, -2.0, 0.3])
approx = numerical_gradient(lambda x: f_quadratic(x, eigenvalues_check), x_check)
got = grad_quadratic(x_check, eigenvalues_check)
assert np.allclose(approx, got, atol=1e-3), f"expected ~{approx}, got {got}"

print("grad_quadratic() looks correct on this case.")


### Visualizing a trajectory in 2D

`plot_trajectory_2d` (given) draws a function of a 2D point as contour lines, with a trajectory on top. For example:

```python
plot_trajectory_2d(lambda p: f_quadratic(p, eigenvalues), trajectory, xlim=(-10, 10), ylim=(-10, 10))
```


In [ ]:
def plot_trajectory_2d(f, trajectory, xlim=None, ylim=None, levels=30, log_scale=False, minimum=None, n_grid=100):
    """Draw a function of a 2D point as contour lines, with a trajectory on top.

    f          : function of a length-2 array, returning a number
    trajectory : list of 2D points, as returned by compute_gradient_trajectory
    xlim, ylim : plotted region (default: a box around the trajectory)
    levels     : number of contour levels
    log_scale  : draw contours of log(1 + f) instead of f, for functions spanning many orders of magnitude
    minimum    : (x, y) of the known minimum, drawn as a star
    n_grid     : grid resolution
    """
    traj = np.array(trajectory)
    if xlim is None:
        xlim = (traj[:, 0].min() - 1, traj[:, 0].max() + 1)
    if ylim is None:
        ylim = (traj[:, 1].min() - 1, traj[:, 1].max() + 1)

    # evaluate f on a grid
    xs = np.linspace(xlim[0], xlim[1], n_grid)
    ys = np.linspace(ylim[0], ylim[1], n_grid)
    XX, YY = np.meshgrid(xs, ys)
    ZZ = np.array([[f(np.array([x, y])) for x in xs] for y in ys])
    if log_scale:
        ZZ = np.log1p(ZZ)

    plt.figure(figsize=(7, 6))
    plt.contour(XX, YY, ZZ, levels=levels, cmap="viridis")
    plt.plot(traj[:, 0], traj[:, 1], "o-", color="red", markersize=3, linewidth=1, label="trajectory")
    plt.plot(*traj[0], "s", color="green", markersize=8, label="start")
    plt.plot(*traj[-1], "X", color="red", markersize=10, label="end")
    if minimum is not None:
        plt.plot(*minimum, "*", color="gold", markeredgecolor="black", markersize=14, label="minimum")
    plt.xlim(xlim)
    plt.ylim(ylim)
    plt.xlabel("x[0]")
    plt.ylabel("x[1]")
    plt.legend()
    plt.show()


**Exercise:** at `d=2`, with `eigenvalues = make_eigenvalues(2)`, run `compute_gradient_trajectory` from `x0 = np.array([8.0, 8.0])` and visualize the trajectory with `plot_trajectory_2d`.


In [ ]:
# your code here

**Optional exercise:** draw the same trajectory on a 3D surface of the function. Evaluate the function on a grid, then use `ax.plot_surface` for the surface and `ax.plot` for the trajectory, with `ax = plt.figure().add_subplot(projection="3d")`.


In [ ]:
# your code here


**Exercise:** repeat the descent for `d = 2`, `d = 10`, `d = 100` and `d = 1000`, from `x0 = np.full(d, 8.0)` and with the **same learning rate** every time, chosen once for `d = 2`. Plot `f_quadratic(x, eigenvalues)` against the step number for each `d`, on the same axes.

**Discussion question:** why do you think the descent gets harder as `d` grows?


In [ ]:
# your code here

### Optional: the Rosenbrock function

A classic test function for comparing optimizers:

$$f(x, y) = (a - x)^2 + b\,(y - x^2)^2, \qquad a = 1,\ b = 100$$

It has a single minimum at $(1, 1)$, inside a long, narrow, curved valley.

**Optional exercise:** implement `rosenbrock(xy)`, derive and implement its gradient as `rosenbrock_grad(xy)` (checked against `numerical_gradient`), then run `compute_gradient_trajectory` from `(-1.5, 2.0)` and visualize the trajectory with `plot_trajectory_2d`, for instance with `xlim=(-2, 2)`, `ylim=(-1, 3)`, `log_scale=True` and `minimum=(1, 1)`. Expect to need a small learning rate and many steps.


In [ ]:
def rosenbrock(xy):
    # your code here
    ...


def rosenbrock_grad(xy):
    # your code here
    ...


# your self-check and gradient descent run here

---
## 4. Optimizing parameters, not inputs

In deep learning you usually don't optimize the inputs; you optimize the model's parameters, and the data enters through the **loss**: the average error of the model's predictions on the dataset. Changing the parameters changes the loss; the data stays fixed. Here the model has a single parameter `theta`, so the whole loss can be plotted: a one-dimensional slice of what, for a real model, is a landscape with millions of dimensions.

The cell below generates noisy samples from `sin(theta_true * x)`. The goal is to recover `theta_true`.


In [ ]:
np.random.seed(0)
theta_true = 1.5
x_data = np.random.uniform(-6, 6, 40)
y_data = np.sin(theta_true * x_data) + np.random.randn(40) * 0.05

plt.scatter(x_data, y_data, s=15)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Noisy samples from sin(theta_true * x)")
plt.show()

**Exercise:** implement `loss(theta, x_data, y_data)`, the mean squared error between `sin(theta * x_data)` and `y_data`, $\frac{1}{N}\sum_i (\sin(\theta x_i) - y_i)^2$. Then derive its gradient with respect to `theta` by hand (`x_data` and `y_data` are fixed; only `theta` varies), implement it as `loss_grad(theta, x_data, y_data)`, and check it against `numerical_derivative`.


In [ ]:
def loss(theta, x_data, y_data):
    # your code here
    ...


def loss_grad(theta, x_data, y_data):
    # your code here
    ...


# self-check
for theta in [-1.0, 0.3, 1.5, 2.0]:
    approx = numerical_derivative(lambda t: loss(t, x_data, y_data), theta)
    got = loss_grad(theta, x_data, y_data)
    assert abs(approx - got) < 1e-3, f"at theta={theta}, expected ~{approx}, got {got}"

print("loss() and loss_grad() look correct on these cases.")

**Exercise:** plot `loss(theta, x_data, y_data)` for `theta` from `-4` to `4`.


In [ ]:
# your code here

**Exercise:** run `compute_gradient_trajectory` to minimize `loss` over `theta`, from a few different starting values, using `loss_grad` as the gradient, with `x_data` and `y_data` held fixed. For each run, plot the trajectory on the loss curve with `plot_trajectory_1d`, and the fitted curve `sin(theta * x)` over the data points.

In [ ]:
# your code here

---
## 5. Classification: a sigmoid neuron against the perceptron

Lab 3's perceptron outputs a hard `-1` or `+1`. A sigmoid neuron instead predicts $\sigma(w \cdot x + b)$ from weights `w` and bias `b`, where $\sigma(z) = \frac{1}{1 + e^{-z}}$, a value between `0` and `1`. Same idea as part 4, for a model with several parameters.

**Exercise:** pack `w` and `b` into one parameter vector, `params = np.array([w1, w2, b])`. Implement `neuron_loss(params, x_data, y_data)`, the mean squared error between $\sigma(w \cdot x + b)$ and `y_data` over the whole dataset, and derive and implement its gradient as `neuron_loss_grad(params, x_data, y_data)`, checked against `numerical_gradient`.


In [ ]:
def neuron_loss(params, x_data, y_data):
    # your code here
    ...


def neuron_loss_grad(params, x_data, y_data):
    # your code here
    ...


# self-check
np.random.seed(0)
x_check = np.random.randn(10, 2)
y_check = (np.random.rand(10) < 0.5).astype(float)
params_check = np.array([0.5, -0.3, 0.1])

approx = numerical_gradient(lambda p: neuron_loss(p, x_check, y_check), params_check)
got = neuron_loss_grad(params_check, x_check, y_check)
assert np.allclose(approx, got, atol=1e-3), f"expected ~{approx}, got {got}"

print("neuron_loss() and neuron_loss_grad() look correct on this case.")


### Training on Lab 3's data

The cells below regenerate Lab 3's two datasets: the linearly separable one and the overlapping clusters, which is closer to real data. The neuron uses labels `0` and `1`, the perceptron its own `-1` and `+1`, on the same points.


In [ ]:
n_samples = 200

X_sep = np.random.randn(n_samples, 2) * 3.0
y_sep_perceptron = np.where(X_sep[:, 0] + X_sep[:, 1] >= 0, 1.0, -1.0)
X_sep[y_sep_perceptron == 1] += [.2, .2]
X_sep[y_sep_perceptron == -1] -= [.2, .2]
y_sep_neuron = np.where(y_sep_perceptron == 1, 1.0, 0.0)

cluster_std = 2.0
n_per_class = 50
X_overlap = np.concatenate([
    np.random.randn(n_per_class, 2) * cluster_std + np.array([2.0, 2.0]),
    np.random.randn(n_per_class, 2) * cluster_std + np.array([-2.0, -2.0]),
])
y_overlap_perceptron = np.concatenate([np.ones(n_per_class), -np.ones(n_per_class)])
y_overlap_neuron = np.concatenate([np.ones(n_per_class), np.zeros(n_per_class)])

# interleave the two classes
order = np.random.permutation(2 * n_per_class)
X_overlap = X_overlap[order]
y_overlap_perceptron = y_overlap_perceptron[order]
y_overlap_neuron = y_overlap_neuron[order]

**Given:** Lab 3's perceptron training loop, which also records the parameters `[w1, w2, b]` after each epoch.


In [ ]:
from minitorch.layer import Perceptron


def train_perceptron(X, y, epochs):
    model = Perceptron(num_features=X.shape[1])
    mistakes_per_epoch = []
    params_per_epoch = []
    for epoch in range(epochs):
        mistakes = 0
        for xi, yi in zip(X, y):
            if model.learn(xi, yi):
                mistakes += 1
        mistakes_per_epoch.append(mistakes)
        params_per_epoch.append(np.append(model.weight, model.bias))
    return mistakes_per_epoch, params_per_epoch


### Visualizing the decision boundary

`plot_boundaries(X, y, params_per_epoch)` (given) draws the data and the decision boundary of each parameter vector `[w1, w2, b]` in `params_per_epoch`, from light gray (first) to black (last). It draws on the current axes.


In [ ]:
def plot_boundaries(X, y, params_per_epoch):
    """Draw the data and the decision boundaries w1*x[0] + w2*x[1] + b = 0, from light gray (first) to black (last).

    Draws on the current axes.

    X                : array of shape (n, 2)
    y                : labels, positive for the positive class (works for 0/1 and -1/+1)
    params_per_epoch : list of parameter vectors [w1, w2, b]
    """
    X = np.asarray(X)
    positive = np.asarray(y) > 0
    plt.scatter(X[positive, 0], X[positive, 1], s=10, label="positive class")
    plt.scatter(X[~positive, 0], X[~positive, 1], s=10, label="negative class")

    xlim = (X[:, 0].min() - 1, X[:, 0].max() + 1)
    ylim = (X[:, 1].min() - 1, X[:, 1].max() + 1)
    XX, YY = np.meshgrid(np.linspace(xlim[0], xlim[1], 200), np.linspace(ylim[0], ylim[1], 200))
    n = len(params_per_epoch)
    for i, (w1, w2, b) in enumerate(params_per_epoch):
        if w1 == 0 and w2 == 0:
            continue
        shade = 0.8 - 0.8 * i / max(n - 1, 1)
        plt.contour(XX, YY, w1 * XX + w2 * YY + b, levels=[0], colors=[str(shade)], linewidths=1)

    plt.xlim(xlim)
    plt.ylim(ylim)
    plt.xlabel("x[0]")
    plt.ylabel("x[1]")
    plt.legend()


**Exercise:** write `train_neuron(X, y, lr, epochs)`, which trains the neuron one example at a time, like the perceptron: starting from `params` set to zeros, go through the examples in order in each epoch and take a gradient step on the loss of that single example (using `neuron_loss_grad` on arrays of length 1). This is **stochastic gradient descent**. Return the loss on the whole dataset after each epoch, and the parameters after each epoch.

Then train both models on the separable dataset for 20 epochs, and plot for each model its curve (the perceptron's mistakes per epoch, the neuron's loss per epoch) and its decision boundaries over the last 10 epochs with `plot_boundaries`. The mistake count only counts misclassified points; the loss measures how far the predictions are from the labels.


In [ ]:
def train_neuron(X, y, lr, epochs):
    # your code here
    ...


# your training calls and plots here


**Exercise:** train both models on the overlapping dataset for 100 epochs, and plot the same figures, with the decision boundaries over the last 20 epochs.

**Discussion question:** Compare the behavior of the perceptron and the sigmoid neuron in the last epochs, and explain the differences.


In [ ]:
# your code here

---
## Key takeaways

- The learning rate has a limit set by the landscape.
- The starting point decides which minimum you end up in.
- Steepness that varies between regions or directions slows gradient descent down or breaks it.
- Gradient descent minimizes a loss over a model's parameters, whatever the model.
